# Re-Verification — Reproductive Entities (OpenAI Batch API)

Runs `prompt_verification_reprod_new.md` on every JSON file in `dataset_ds_reverif/`  
using the **OpenAI Batch API** (async, 50 % cost discount, 24 h window).

### Workflow
| Cell | Action |
|------|--------|
| **1 — Config** | Set paths, model, parameters |
| **2 — Submit** | Build JSONL, upload file, create batch job |
| **3 — Poll**   | Check batch status until complete |
| **4 — Apply**  | Download results, write `verification_reprod` into each JSON file |
| **5 — Summary**| PASS / FAIL / GENERATION_FAILURE counts |
| **6 — FAIL detail** | List files with critical errors |
| **7 — GENERATION_FAILURE detail** | List generation failures |

**Run cells 2 → 3 → 4 in order.  
Cell 3 can be re-run repeatedly until status = `completed`.**

In [ ]:
import json, os, time, urllib.request, urllib.error, io
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../../..").resolve() / ".env")

# ── Configuration ─────────────────────────────────────────────────────────────
API_KEY    = os.environ.get("OPENAI_API_KEY")

# gpt-4.1 — OpenAI's best model for precise instruction-following and JSON output.
# Supports response_format=json_object, large context, and Batch API.
MODEL      = "gpt-4.1"

INPUT_DIR  = Path("../dataset_ds_reverif")
PROMPT_FILE = Path("../verif_prompts/prompt_verification_reprod_new.md")

# Local file that stores the batch_id so you can close the notebook and resume
BATCH_ID_FILE = Path("../dataset_ds_reverif/_batch_id.txt")

# ── Generation parameters ─────────────────────────────────────────────────────
TEMPERATURE = 0.0    # deterministic — essential for reproducible annotation audit
MAX_TOKENS  = 8192   # corrected_text_final can be long; avoid truncation

# ── OpenAI API base ───────────────────────────────────────────────────────────
OAI_BASE   = "https://api.openai.com/v1"

# ─────────────────────────────────────────────────────────────────────────────
if not API_KEY:
    raise ValueError("OPENAI_API_KEY not set — check your .env file.")

SYSTEM_PROMPT = PROMPT_FILE.read_text(encoding="utf-8")
json_files    = sorted(INPUT_DIR.glob("*.json"))
# Exclude the internal batch-id tracker if it ever ends up as .json
json_files    = [f for f in json_files if not f.name.startswith("_")]

already_done  = [f for f in json_files
                 if json.loads(f.read_text()).get("verification_reprod") is not None]
pending       = [f for f in json_files
                 if json.loads(f.read_text()).get("verification_reprod") is None
                 and json.loads(f.read_text()).get("corrected_text")]

print(f"Model            : {MODEL}")
print(f"Temperature      : {TEMPERATURE}")
print(f"Max tokens       : {MAX_TOKENS}")
print(f"Prompt file      : {PROMPT_FILE.name} ({len(SYSTEM_PROMPT):,} chars)")
print(f"Total files      : {len(json_files)}")
print(f"Already verified : {len(already_done)}")
print(f"Pending (will batch): {len(pending)}")

## Cell 2 — Submit Batch
Builds the JSONL request file, uploads it to OpenAI Files API, and creates the batch job.  
**Only run this once.** The batch ID is saved to `_batch_id.txt` so you can resume later.

In [ ]:
def oai_request(method: str, path: str, data=None, headers_extra: dict = None,
                raw_body: bytes = None, content_type: str = "application/json") -> dict:
    """Thin wrapper around urllib for OpenAI REST calls."""
    url = f"{OAI_BASE}{path}"
    headers = {"Authorization": f"Bearer {API_KEY}"}
    if headers_extra:
        headers.update(headers_extra)

    if raw_body is not None:
        body = raw_body
    elif data is not None:
        body = json.dumps(data).encode()
        headers["Content-Type"] = content_type
    else:
        body = None

    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode())


def oai_download(path: str) -> bytes:
    """Download raw bytes (for batch output file)."""
    url = f"{OAI_BASE}{path}"
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {API_KEY}"}, method="GET")
    with urllib.request.urlopen(req, timeout=180) as resp:
        return resp.read()


def upload_multipart_jsonl(jsonl_bytes: bytes, filename: str) -> str:
    """Upload a JSONL file via multipart/form-data and return the file_id."""
    boundary = b"----OAIBoundary7f3k9x"
    body = (
        b"--" + boundary + b"\r\n"
        b'Content-Disposition: form-data; name="purpose"\r\n\r\n'
        b"batch\r\n"
        b"--" + boundary + b"\r\n"
        + f'Content-Disposition: form-data; name="file"; filename="{filename}"\r\n'.encode()
        + b"Content-Type: application/jsonl\r\n\r\n"
        + jsonl_bytes + b"\r\n"
        b"--" + boundary + b"--\r\n"
    )
    ct = f"multipart/form-data; boundary={boundary.decode()}"
    url = f"{OAI_BASE}/files"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": ct,
    }
    req = urllib.request.Request(url, data=body, headers=headers, method="POST")
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode())["id"]


# Guard: don't re-submit if a batch is already running
if BATCH_ID_FILE.exists():
    existing = BATCH_ID_FILE.read_text().strip()
    print(f"Batch already submitted: {existing}")
    print("Skip to Cell 3 (Poll) or Cell 4 (Apply) if completed.")
    print("Delete _batch_id.txt to start a fresh batch.")
else:
    if not pending:
        print("Nothing to submit — all files already verified.")
    else:
        # ── Build JSONL ────────────────────────────────────────────────────────
        lines = []
        for fp in pending:
            data = json.loads(fp.read_text(encoding="utf-8"))
            user_payload = json.dumps({
                "id":             data.get("id"),
                "specie":         data.get("specie"),
                "specie_en":      data.get("specie_en"),
                "corrected_text": data.get("corrected_text"),
            }, ensure_ascii=False)

            request_obj = {
                "custom_id": fp.stem,   # filename without .json — used to match results back
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL,
                    "temperature": TEMPERATURE,
                    "max_tokens": MAX_TOKENS,
                    "response_format": {"type": "json_object"},
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_payload},
                    ],
                },
            }
            lines.append(json.dumps(request_obj, ensure_ascii=False))

        jsonl_bytes = "\n".join(lines).encode("utf-8")
        print(f"JSONL built: {len(lines)} requests, {len(jsonl_bytes):,} bytes")

        # ── Upload JSONL file ──────────────────────────────────────────────────
        print("Uploading batch input file...", end=" ", flush=True)
        file_id = upload_multipart_jsonl(jsonl_bytes, "reverif_batch.jsonl")
        print(f"uploaded → {file_id}")

        # ── Create batch job ───────────────────────────────────────────────────
        print("Creating batch job...", end=" ", flush=True)
        batch = oai_request("POST", "/batches", data={
            "input_file_id":  file_id,
            "endpoint":       "/v1/chat/completions",
            "completion_window": "24h",
            "metadata": {"description": "DEB reverif reprod — gpt-4.1"},
        })
        batch_id = batch["id"]
        BATCH_ID_FILE.write_text(batch_id)
        print(f"created → {batch_id}")
        print(f"Status : {batch['status']}")
        print(f"Batch ID saved to: {BATCH_ID_FILE}")
        print("\nNow run Cell 3 to poll status.")

## Cell 3 — Poll Status
Re-run this cell until `status = completed`.  
OpenAI typically completes batches of ~100 requests in a few minutes to a few hours.

In [ ]:
if not BATCH_ID_FILE.exists():
    print("No batch ID found. Run Cell 2 first.")
else:
    batch_id = BATCH_ID_FILE.read_text().strip()
    batch    = oai_request("GET", f"/batches/{batch_id}")
    status   = batch["status"]
    counts   = batch.get("request_counts", {})

    print(f"Batch ID : {batch_id}")
    print(f"Status   : {status}")
    print(f"Requests : total={counts.get('total','-')}  "
          f"completed={counts.get('completed','-')}  "
          f"failed={counts.get('failed','-')}")

    if status == "completed":
        print("\nBatch complete! Run Cell 4 to apply results.")
    elif status in ("failed", "expired", "cancelled"):
        print(f"\nBatch ended with status '{status}'. Check OpenAI dashboard for details.")
        errors = batch.get("errors", {})
        if errors:
            print(json.dumps(errors, indent=2))
    else:
        print("\nStill processing — re-run this cell in a few minutes.")

## Cell 4 — Apply Results
Downloads the output JSONL and writes `verification_reprod` into each source JSON file.  
Safe to re-run: already-applied files are skipped.

In [ ]:
if not BATCH_ID_FILE.exists():
    print("No batch ID found. Run Cell 2 first.")
else:
    batch_id = BATCH_ID_FILE.read_text().strip()
    batch    = oai_request("GET", f"/batches/{batch_id}")
    status   = batch["status"]

    if status != "completed":
        print(f"Batch status is '{status}' — not yet completed. Run Cell 3 to check again.")
    else:
        output_file_id = batch.get("output_file_id")
        error_file_id  = batch.get("error_file_id")

        # ── Download output JSONL ──────────────────────────────────────────────
        print(f"Downloading output file {output_file_id}...", end=" ", flush=True)
        raw_output = oai_download(f"/files/{output_file_id}/content")
        print(f"{len(raw_output):,} bytes")

        # ── Parse output lines ─────────────────────────────────────────────────
        applied   = 0
        skipped   = 0
        api_errors = 0
        parse_errors = 0

        for line in raw_output.decode("utf-8").splitlines():
            if not line.strip():
                continue
            result = json.loads(line)
            custom_id = result["custom_id"]           # stem of the filename
            filepath  = INPUT_DIR / f"{custom_id}.json"

            if not filepath.exists():
                print(f"  [WARN] File not found for custom_id '{custom_id}'")
                continue

            data = json.loads(filepath.read_text(encoding="utf-8"))

            if data.get("verification_reprod") is not None:
                skipped += 1
                continue

            # Check for API-level error in the result
            if result.get("error"):
                print(f"  [API-ERROR] {custom_id}: {result['error']}")
                api_errors += 1
                continue

            # Extract the model's text content
            response_body = result.get("response", {}).get("body", {})
            raw_content   = response_body.get("choices", [{}])[0].get("message", {}).get("content", "")

            # Parse the JSON response
            try:
                verification = json.loads(raw_content)
            except json.JSONDecodeError:
                stripped = raw_content.strip().removeprefix("```json").removesuffix("```").strip()
                try:
                    verification = json.loads(stripped)
                except json.JSONDecodeError:
                    print(f"  [PARSE-ERROR] {custom_id}: could not parse model output")
                    parse_errors += 1
                    continue

            data["verification_reprod"] = verification
            filepath.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

            verdict  = verification.get("verdict", "?")
            critical = verification.get("critical_errors", "?")
            usage    = response_body.get("usage", {})
            print(f"  {custom_id}  →  {verdict} | {critical} critical "
                  f"| {usage.get('prompt_tokens','?')} in / {usage.get('completion_tokens','?')} out")
            applied += 1

        print(f"\nDone: {applied} applied, {skipped} already done, "
              f"{api_errors} API errors, {parse_errors} parse errors")

        # ── Show API errors file if present ────────────────────────────────────
        if error_file_id:
            print(f"\nError file present ({error_file_id}) — downloading...")
            raw_errors = oai_download(f"/files/{error_file_id}/content")
            for line in raw_errors.decode("utf-8").splitlines():
                if line.strip():
                    print(" ", json.loads(line))

## Cell 5 — Summary

In [ ]:
from collections import Counter

counts = Counter()
for filepath in sorted(INPUT_DIR.glob("*.json")):
    if filepath.name.startswith("_"):
        continue
    data = json.loads(filepath.read_text(encoding="utf-8"))
    if not data.get("corrected_text"):
        counts["NO_TEXT"] += 1
        continue
    v = data.get("verification_reprod")
    if v is None:
        counts["NOT_VERIFIED"] += 1
    else:
        counts[v.get("verdict", "UNKNOWN")] += 1

total = sum(counts.values())
print(f"=== Re-Verification Summary ({total} files) ===")
for verdict, count in counts.most_common():
    print(f"  {verdict}: {count}")

## Cell 6 — FAIL detail

In [ ]:
fails = []
for filepath in sorted(INPUT_DIR.glob("*.json")):
    if filepath.name.startswith("_"):
        continue
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification_reprod")
    if v and v.get("verdict") == "FAIL":
        fails.append({
            "file":            filepath.name,
            "specie":          data.get("specie"),
            "critical_errors": v.get("critical_errors"),
            "error_codes":     [e.get("code") for e in v.get("errors", [])],
        })

print(f"FAIL ({len(fails)} files):")
for r in fails:
    print(f"  {r['file']}  —  {r['specie']}  —  {r['critical_errors']} critical  —  {r['error_codes']}")

## Cell 7 — GENERATION_FAILURE detail

In [ ]:
failures = []
for filepath in sorted(INPUT_DIR.glob("*.json")):
    if filepath.name.startswith("_"):
        continue
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification_reprod")
    if v and v.get("verdict") == "GENERATION_FAILURE":
        failures.append({"file": filepath.name, "specie": data.get("specie")})

print(f"GENERATION_FAILURE ({len(failures)} files):")
for r in failures:
    print(f"  {r['file']}  —  {r['specie']}")